# 🎯 Auditor BI-RADS — RoBERTa Clínico + Focal Loss

Experimento **controlado** que parte del mejor modelo hasta ahora (RoBERTa clínico, F1-macro test = 0,5871) y cambia **únicamente la función de pérdida**: de *cross-entropy* ponderada a **Focal Loss**.

**Qué hace focal loss:** reduce el peso de los ejemplos fáciles (clases frecuentes que el modelo ya acierta) y concentra el aprendizaje en los **difíciles** (las clases de riesgo poco frecuentes). Combina el factor focal `(1 - p)^γ` con los pesos de clase como `α`.

Todo lo demás (modelo, split, augmentación, hiperparámetros, semilla 42) es idéntico, para aislar el efecto de la pérdida.

## 1 · Configuración y librerías

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
from collections import Counter
import os, sys
sys.path.append('..')
from src.preprocessing import limpiar_texto, MAX_LENGTH

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():           DEVICE = 'cuda'
elif torch.backends.mps.is_available(): DEVICE = 'mps'
else:                                   DEVICE = 'cpu'

MODELO     = 'PlanTL-GOB-ES/roberta-base-biomedical-clinical-es'
MODELO_ALT = 'BSC-LT/roberta-base-biomedical-clinical-es'
GAMMA = 2.0   # factor focal

print(f"PyTorch: {torch.__version__} | Dispositivo: {DEVICE}")
print(f"Modelo: {MODELO} | Focal Loss gamma={GAMMA}")

PyTorch: 2.12.0 | Dispositivo: mps
Modelo: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es | Focal Loss gamma=2.0


## 2 · Carga del dataset curado

In [2]:
df = pd.read_csv('../data/processed/dataset_clean.csv', encoding='utf-8')
print(f"Informes: {len(df)} | Columnas: {df.shape[1]}")

Informes: 4357 | Columnas: 19


## 3 · División train / validación / prueba (solo observaciones)

In [3]:
X = df['auditor_input'].values
y = df['BI-RADS'].values
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.176, random_state=SEED, stratify=y_tv)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 3051 | Val: 652 | Test: 654


## 4 · Pesos de clase (alpha para focal loss)

In [4]:
clases = np.unique(y_train)
pesos = compute_class_weight(class_weight='balanced', classes=clases, y=y_train)
peso_vec = np.ones(7, dtype=np.float32)
for c, w in zip(clases, pesos): peso_vec[c] = w
alpha = torch.tensor(peso_vec, dtype=torch.float32).to(DEVICE)
print("Alpha por clase (0..6):", np.round(peso_vec, 2))

Alpha por clase (0..6): [  0.64   1.04   0.24   7.15  12.11  36.32 145.29]


## 5 · Augmentación de clases minoritarias (SOLO train)

In [5]:
def aumentar_texto(texto):
    variaciones = [
        ('bordes irregulares', 'márgenes irregulares'),
        ('bordes espiculados', 'márgenes espiculados'),
        ('bordes mal definidos', 'límites imprecisos'),
        ('imagen nodular', 'nódulo'), ('nódulo', 'imagen nodular'),
        ('lesión nodular', 'nódulo'),
        ('heterogéneamente densas', 'de densidad heterogénea'),
        ('muy densas', 'extremadamente densas'),
        ('calcificaciones sospechosas', 'depósitos cálcicos sospechosos'),
        ('microcalcificaciones', 'calcificaciones puntiformes agrupadas'),
        ('mama derecha', 'hemimama derecha'), ('mama izquierda', 'hemimama izquierda'),
        ('se visualiza', 'se observa'), ('se observa', 'se visualiza'),
        ('se evidencia', 'se observa'),
    ]
    t = texto
    for orig, rep in random.sample(variaciones, min(3, len(variaciones))):
        if orig in t: t = t.replace(orig, rep, 1)
    return t

mask_min = np.isin(y_train, [3, 4, 5, 6])
X_min, y_min = X_train[mask_min], y_train[mask_min]
textos_aug, labels_aug = [], []
for txt, lab in zip(X_min, y_min):
    for _ in range(3):
        textos_aug.append(aumentar_texto(txt)); labels_aug.append(lab)

X_train_aug = np.concatenate([X_train, np.array(textos_aug)])
y_train_aug = np.concatenate([y_train, np.array(labels_aug)])
idx = np.random.permutation(len(X_train_aug))
X_train_aug, y_train_aug = X_train_aug[idx], y_train_aug[idx]
print(f"Train aumentado: {len(X_train_aug)} (sin fuga de val/test)")

Train aumentado: 3387 (sin fuga de val/test)


## 6 · Tokenizador (RoBERTa clínico)

In [6]:
try:
    tokenizer = AutoTokenizer.from_pretrained(MODELO)
except Exception as e:
    print(f"Fallback a {MODELO_ALT} ({e})"); MODELO = MODELO_ALT
    tokenizer = AutoTokenizer.from_pretrained(MODELO)
print("Tokenizador cargado.")

Tokenizador cargado.


## 7 · Dataset y DataLoader

In [7]:
class BIRADSDataset(Dataset):
    def __init__(self, textos, labels, tokenizer, max_length=MAX_LENGTH):
        self.textos=list(textos); self.labels=list(labels); self.tok=tokenizer; self.max_length=max_length
    def __len__(self): return len(self.textos)
    def __getitem__(self, i):
        enc = self.tok(str(self.textos[i]), truncation=True, padding='max_length',
                       max_length=self.max_length, return_tensors='pt', return_token_type_ids=False)
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.labels[i], dtype=torch.long)}

BATCH = 16
train_loader = DataLoader(BIRADSDataset(X_train_aug, y_train_aug, tokenizer), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(BIRADSDataset(X_val,  y_val,  tokenizer), batch_size=BATCH)
test_loader  = DataLoader(BIRADSDataset(X_test, y_test, tokenizer), batch_size=BATCH)
print(f"Batches -> train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}")

Batches -> train: 212 | val: 41 | test: 41


## 8 · Arquitectura del modelo

In [8]:
class AuditorRoBERTa(nn.Module):
    def __init__(self, modelo, n_classes=7, dropout=0.5):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(modelo)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, n_classes)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls))

model = AuditorRoBERTa(MODELO, n_classes=7, dropout=0.5).to(DEVICE)
print(f"Modelo cargado en {DEVICE}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado en mps


## 9 · Focal Loss y configuración del entrenamiento

**Focal Loss** (Lin et al., 2017): `FL = -α (1 - p_t)^γ · log(p_t)`, donde `p_t` es la probabilidad asignada a la clase correcta. Cuando el modelo ya acierta con alta confianza (`p_t→1`), el factor `(1-p_t)^γ` reduce la pérdida de ese ejemplo, forzando al modelo a enfocarse en los casos difíciles. `α` son los pesos de clase.

In [9]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha   # tensor [n_classes] o None
        self.gamma = gamma
    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=1)
        logp_t = logp.gather(1, target.unsqueeze(1)).squeeze(1)
        p_t = logp_t.exp()
        loss = -((1 - p_t) ** self.gamma) * logp_t
        if self.alpha is not None:
            loss = self.alpha.gather(0, target) * loss
        return loss.mean()

LR = 3e-5; EPOCHS = 15; PATIENCE = 4
criterio  = FocalLoss(alpha=alpha, gamma=GAMMA)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader)*EPOCHS)
print(f"Pérdida: Focal Loss (gamma={GAMMA}, alpha=pesos de clase) | LR={LR} | Épocas={EPOCHS}")

Pérdida: Focal Loss (gamma=2.0, alpha=pesos de clase) | LR=3e-05 | Épocas=15


## 10 · Entrenamiento

In [10]:
def correr_epoca(loader, entrenar=True):
    model.train() if entrenar else model.eval()
    preds, reales, perdida_total = [], [], 0.0
    with torch.set_grad_enabled(entrenar):
        for batch in loader:
            ids=batch['input_ids'].to(DEVICE); mask=batch['attention_mask'].to(DEVICE)
            labels=batch['labels'].to(DEVICE)
            if entrenar: optimizer.zero_grad()
            logits=model(ids, mask); loss=criterio(logits, labels)
            if entrenar: loss.backward(); optimizer.step(); scheduler.step()
            perdida_total+=loss.item()
            preds.extend(logits.argmax(1).cpu().numpy()); reales.extend(labels.cpu().numpy())
    return perdida_total/len(loader), f1_score(reales, preds, average='macro', zero_division=0)

os.makedirs('../results', exist_ok=True)
RUTA='../results/mejor_auditor_roberta_focal.pt'
print("🏋️  Entrenando (RoBERTa clínico + Focal Loss)")
print("Época | Train Loss | Val Loss | Train F1 | Val F1")
mejor_f1, sin_mejora = 0.0, 0
for ep in range(1, EPOCHS+1):
    tl,tf1=correr_epoca(train_loader,True); vl,vf1=correr_epoca(val_loader,False)
    marca=""
    if vf1>mejor_f1: mejor_f1,sin_mejora=vf1,0; torch.save(model.state_dict(),RUTA); marca="  <- mejor"
    else: sin_mejora+=1
    print(f"  {ep:2d}  |  {tl:.4f}  |  {vl:.4f}  |  {tf1:.4f}  |  {vf1:.4f}{marca}")
    if sin_mejora>=PATIENCE: print(f"Early stopping en época {ep}"); break
print(f"🏆 Mejor Val F1 = {mejor_f1:.4f}")

🏋️  Entrenando (RoBERTa clínico + Focal Loss)
Época | Train Loss | Val Loss | Train F1 | Val F1
   1  |  2.9833  |  0.9165  |  0.0567  |  0.2564  <- mejor
   2  |  1.4563  |  0.6596  |  0.3002  |  0.3444  <- mejor
   3  |  0.6597  |  0.4845  |  0.5650  |  0.5396  <- mejor
   4  |  0.3428  |  0.4958  |  0.7558  |  0.5754  <- mejor
   5  |  0.2150  |  0.5925  |  0.8133  |  0.5714
   6  |  0.1124  |  0.5687  |  0.8813  |  0.6760  <- mejor
   7  |  0.0907  |  0.6909  |  0.8994  |  0.5851
   8  |  0.0710  |  0.7348  |  0.9213  |  0.5784
   9  |  0.0661  |  0.8291  |  0.9248  |  0.5854
  10  |  0.0502  |  0.9558  |  0.9394  |  0.5743
Early stopping en época 10
🏆 Mejor Val F1 = 0.6760


## 11 · Evaluación en test

In [11]:
model.load_state_dict(torch.load(RUTA, map_location=DEVICE)); model.eval()
preds, reales = [], []
with torch.no_grad():
    for batch in test_loader:
        ids=batch['input_ids'].to(DEVICE); mask=batch['attention_mask'].to(DEVICE)
        preds.extend(model(ids,mask).argmax(1).cpu().numpy()); reales.extend(batch['labels'].numpy())
acc=(np.array(preds)==np.array(reales)).mean()
f1m=f1_score(reales, preds, average='macro', zero_division=0)
print("="*55)
print("📊 AUDITOR RoBERTa CLÍNICO + FOCAL LOSS — TEST SET")
print("="*55)
print(f"  Accuracy:         {acc:.4f} ({acc*100:.2f}%)")
print(f"  F1-Score (macro): {f1m:.4f}")
print("="*55)
print("\nReporte por clase:")
print(classification_report(reales, preds, target_names=[f'BI-RADS {i}' for i in range(7)], zero_division=0))

📊 AUDITOR RoBERTa CLÍNICO + FOCAL LOSS — TEST SET
  Accuracy:         0.8303 (83.03%)
  F1-Score (macro): 0.5070

Reporte por clase:
              precision    recall  f1-score   support

   BI-RADS 0       0.88      0.52      0.66       145
   BI-RADS 1       0.94      0.98      0.96        89
   BI-RADS 2       0.97      0.92      0.94       396
   BI-RADS 3       0.15      0.92      0.26        13
   BI-RADS 4       0.17      0.38      0.23         8
   BI-RADS 5       0.50      0.50      0.50         2
   BI-RADS 6       0.00      0.00      0.00         1

    accuracy                           0.83       654
   macro avg       0.52      0.60      0.51       654
weighted avg       0.92      0.83      0.86       654



## 12 · Comparación de los tres auditores

In [12]:
print("F1-macro en test (sin fuga, mismo split/semilla 42):")
print(f"   BETO (español general):              0.4773")
print(f"   RoBERTa clínico + cross-entropy:     0.5871")
print(f"   RoBERTa clínico + focal loss:        {f1m:.4f}")
mejor = max(0.4773, 0.5871, f1m)
print(f"\n   Mejor configuración: {'focal loss' if f1m==mejor else 'cross-entropy' if 0.5871==mejor else 'BETO'}")

F1-macro en test (sin fuga, mismo split/semilla 42):
   BETO (español general):              0.4773
   RoBERTa clínico + cross-entropy:     0.5871
   RoBERTa clínico + focal loss:        0.5070

   Mejor configuración: cross-entropy
